In [2]:
#Installing necessary Libraries
!pip install sumy textblob tqdm
!python -m textblob.download_corpora


Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\cvalley\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\cvalley\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\cvalley\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\cvalley\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\cvalley\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\cvalley\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_re

In [4]:
!pip install pdfplumber


In [ ]:
# Import necessary modules
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from textblob import TextBlob
from pathlib import Path
from tqdm.notebook import tqdm
import pdfplumber
import os


class Document:
    def __init__(self, path):
        self.path = Path(path)
        self._dataset = None
        self._summarizer = LexRankSummarizer()

    @property
    def dataset(self):
        if self._dataset is None:
            try:
                if self.path.suffix.lower() == '.pdf':
                    with pdfplumber.open(self.path) as pdf:
                        self._dataset = '\n'.join(page.extract_text() or '' for page in pdf.pages)
                else:
                    with self.path.open("r", encoding='utf-8') as file:
                        self._dataset = file.read()
            except FileNotFoundError:
                raise ValueError("The document file was not found.")
            except PermissionError:
                raise ValueError("Permission denied to access the document file.")
            except Exception as e:
                raise Exception(f"An error occurred while reading the file: {str(e)}")
        return self._dataset

    def summarize(self, sentences_count=7):
        if len(self.dataset.split(".")) < sentences_count:
            raise ValueError("The document contains fewer sentences than requested.")
        parser = PlaintextParser.from_string(self.dataset, Tokenizer("english"))
        summarized_sentences = self._summarizer(parser.document, sentences_count)
        summarized_text = " ".join(str(sentence) for sentence in summarized_sentences)
        return summarized_text

    def analyze_sentiment(self):
        blob = TextBlob(self.dataset)
        polarity = blob.sentiment.polarity
        return "positive" if polarity > 0 else "negative" if polarity < 0 else "neutral"

class TextDocument(Document):
    pass

class WordDocument(Document):
    pass

def process_document(file_path, no_of_sentences):
    file_type = os.path.splitext(file_path)[1]
    document = WordDocument(file_path) if file_type.lower() == ".docx" else TextDocument(file_path)
    
    try:
        summary = document.summarize(no_of_sentences)
        sentiment = document.analyze_sentiment()
        footer = "\n©Bakhtawar Wahid"

        # Print results directly
        print("Summary:")
        print(summary)
        print("\nSentiment Analysis:")
        print(f"Sentiment as analyzed by the model: {sentiment}")
        print(footer)

    except ValueError as e:
        print(f"Error: {str(e)}")
    except FileNotFoundError:
        print("Error: Input file not found.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

# Example usage in the notebook
file_path = input("Enter a valid file path for .txt, .docx, or .pdf file: ").strip('\"')
no_of_sentences = int(input("Enter number of lines for summary: "))

process_document(file_path, no_of_sentences)


In [6]:
!pip install scikit-learn


In [8]:
import tkinter as tk
from tkinter import filedialog, scrolledtext, simpledialog, messagebox
from tkinter.ttk import Progressbar
import threading
import pdfplumber
import os
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from pathlib import Path

class Document:
    def __init__(self, path):
        self.path = Path(path)
        self._dataset = None
        self._summarizer = LexRankSummarizer()

    @property
    def dataset(self):
        if self._dataset is None:
            try:
                if self.path.suffix.lower() == '.pdf':
                    with pdfplumber.open(self.path) as pdf:
                        self._dataset = '\n'.join(page.extract_text() or '' for page in pdf.pages)
                else:
                    with self.path.open("r", encoding='utf-8') as file:
                        self._dataset = file.read()
            except FileNotFoundError:
                raise ValueError("The document file was not found.")
            except PermissionError:
                raise ValueError("Permission denied to access the document file.")
            except Exception as e:
                raise Exception(f"An error occurred while reading the file: {str(e)}")
        return self._dataset

    def summarize(self, sentences_count=7):
        if len(self.dataset.split(".")) < sentences_count:
            raise ValueError("The document contains fewer sentences than requested.")
        parser = PlaintextParser.from_string(self.dataset, Tokenizer("english"))
        summarized_sentences = self._summarizer(parser.document, sentences_count)
        summarized_text = " ".join(str(sentence) for sentence in summarized_sentences)
        return summarized_text

    def analyze_sentiment(self):
        blob = TextBlob(self.dataset)
        polarity = blob.sentiment.polarity
        return "positive" if polarity > 0 else "negative" if polarity < 0 else "neutral"

    def extract_keywords(self, num_keywords=10):
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform([self.dataset])
        feature_array = vectorizer.get_feature_names_out()
        tfidf_sorting = tfidf_matrix.toarray().flatten().argsort()[::-1]
        top_keywords = feature_array[tfidf_sorting][:num_keywords]
        return top_keywords

class GUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Document Analyzer")
        self.root.geometry("800x600")
        

        # Create a button to load documents
        self.load_button = tk.Button(root, text="Load Document", command=self.load_document)
        self.load_button.pack(pady=10)

        # Create a progress bar
        self.progress = Progressbar(root, orient=tk.HORIZONTAL, length=100, mode='indeterminate')
        self.progress.pack(pady=10)

        # Create a scrolled text widget to display the results
        self.result_text = scrolledtext.ScrolledText(root, width=90, height=20)
        self.result_text.pack(pady=10)

    def load_document(self):
        file_path = filedialog.askopenfilename(filetypes=[("All Files", "*.*"), ("Text Files", "*.txt"), ("Word Documents", "*.docx"), ("PDF Files", "*.pdf")])
        if not file_path:
            return

        no_of_sentences = simpledialog.askinteger("Input", "Enter number of lines for summary:", minvalue=1, maxvalue=100)
        if no_of_sentences is None:
            return

        threading.Thread(target=self.process_document, args=(file_path, no_of_sentences)).start()

    def process_document(self, file_path, no_of_sentences):
        self.progress.start()
        try:
            document = Document(file_path)  # Utilizing the Document class to determine file type and process
            summary = document.summarize(no_of_sentences)
            sentiment = document.analyze_sentiment() 
            keywords = document.extract_keywords()

            self.result_text.config(state=tk.NORMAL)
            self.result_text.delete(1.0, tk.END)
            self.result_text.insert(tk.END, f"Summary:\n{summary}\n\n")
            self.result_text.insert(tk.END, f"Sentiment Analysis:\nSentiment as analyzed by the model: {sentiment}\n\n")
            self.result_text.insert(tk.END, f"Keywords:\n{', '.join(keywords)}\n\n")
            self.result_text.config(state=tk.DISABLED)
        except Exception as e:
            messagebox.showerror("Error", str(e))
        finally:
            self.progress.stop()

root = tk.Tk()
app = GUI(root) 
root.mainloop()
